----------------------------------------------------------------------------------------------------

# ***Decoradores***

----------------------------------------------------------------------------------------------------

## ***1. O que é um Decorador?***

Um decorador é uma função (ou objeto chamável) que recebe uma função como parâmetro e retorna uma nova função. Ele permite modificar o comportamento de funções ou métodos sem alterar seu código interno.

Pense em decoradores como "embalagens inteligentes" para suas funções.

A ideia central é: Como se trata de Python, funções em Python são objetos, então podem ser passadas como argumentos, retornadas e atribuídas a variáveis.

### ***1.1. Sintaxe básica***

In [ ]:
@meu_decorador
def minha_funcao():
    pass

É equivalente a:

In [ ]:
def minha_funcao():
    pass

minha_funcao = meu_decorador(minha_funcao)

----------------------------------------------------------------------------------------------------

## ***2. Fundamentos: como funciona***

### ***2.1. funções como objetos em Python***

In [3]:
def saudacao():
    return "Olá!"

# funções podem ser atribuídas a variáveis
outra = saudacao
print(outra())  # "Olá!"

# funções podem ser passadas como argumento
def executar(func):
    return func()

print(executar(saudacao))  # "Olá!"

Olá!
Olá!


outro exemplo:

In [ ]:
def func():
    print("Olá")

print(func) 
nome = func  # pode atribuir a variável
nome()       

<function func at 0x000002E985853560>
Olá


### ***2.1. Estrutura de um decorador simples***

In [ ]:
def meu_decorador(funcao):
    def nova_funcao():
        print("Antes da função")
        funcao()  # chama a função original
        print("Depois da função")
    return nova_funcao  # retorna a nova função 

@meu_decorador
def minha_saudacao():
    print("Olá!")

minha_saudacao()

Antes da função
Olá!
Depois da função


----------------------------------------------------------------------------------------------------

## ***3. Decoradores com parâmetros***

### 3.1. Aceitando *args e **kwargs

In [5]:
def decorador_com_args(funcao):
    def nova_funcao(*args, **kwargs):  # aceita qualquer número de argumentos [web:11]
        print("Antes")
        resultado = funcao(*args, **kwargs)  # passa todos os argumentos
        print("Depois")
        return resultado
    return nova_funcao

@decorador_com_args
def soma(a, b):
    return a + b

print(soma(3, 5))  # 8

Antes
Depois
8


### ***3.2. Decorador com seus próprios parâmetros***

In [ ]:
def decora_repete(n):
    """Decorador que recebe parâmetro"""
    def decorador(funcao):
        def nova_funcao(*args, **kwargs):
            for i in range(n):
                resultado = funcao(*args, **kwargs)
            return resultado
        return nova_funcao
    return decorador

@decora_repete(3)
def saudacao():
    print("Olá!")

saudacao()  # imprime "Olá!" 3 vezes

----------------------------------------------------------------------------------------------------

## ***4. Decoradores práticos comuns***

### ***4.1. Contador de chamadas***

In [ ]:
def contar(funcao):
    contador = 0
    def nova_funcao(*args, **kwargs):
        contador += 1
        print(f"{funcao.__name__} foi chamada {contador} vez(es)")
        return funcao(*args, **kwargs)
    return nova_funcao

@contar
def exibir():
    print("Exibindo...")

exibir()  # exibir foi chamada 1 vez(es)
exibir()  # exibir foi chamada 2 vez(es)

### ***4.2. Cache (Memoização)***

In [ ]:
def cache(funcao):
    memora = {}
    def nova_funcao(*args):
        if args in memora:
            print(f"Retornando cache para {args}")
            return memora[args]
        resultado = funcao(*args)
        memora[args] = resultado
        return resultado
    return nova_funcao

@cache
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(10))
print(fibonacci(10))

Retornando cache para (1,)
Retornando cache para (2,)
Retornando cache para (3,)
Retornando cache para (4,)
Retornando cache para (5,)
Retornando cache para (6,)
Retornando cache para (7,)
Retornando cache para (8,)
55
Retornando cache para (10,)
55


### ***4.3. Validador de tipos***

In [ ]:
def valida_tipos(tipos_esperados):
    def decorador(funcao):
        def nova_funcao(*args):
            for i, (arg, tipo) in enumerate(zip(args, tipos_esperados)):
                if not isinstance(arg, tipo):
                    raise TypeError(f"Argumento {i} deve ser {tipo}, não {type(arg)}")
            return funcao(*args)
        return nova_funcao
    return decorador

@valida_tipos((int, int))
def soma(a, b):
    return a + b

soma(2, 3)      
soma(2, "3")    

### ***4.4. Timer de performance***

In [9]:
import time

def temporizador(funcao):
    def nova_funcao(*args, **kwargs):
        inicio = time.time()
        resultado = funcao(*args, **kwargs)
        fim = time.time()
        print(f"{funcao.__name__} levou {fim - inicio:.4f}s")
        return resultado
    return nova_funcao

@temporizador
def funcao_demorada():
    time.sleep(1)
    print("Feito!")

funcao_demorada()

Feito!
funcao_demorada levou 1.0012s


### ***4.5. Autenticação (como no Flask)***

In [10]:
def requer_autenticacao(funcao):
    def nova_funcao(*args, **kwargs):
        usuario_autenticado = False  # simulação
        if not usuario_autenticado:
            raise PermissionError("Usuário não autenticado")
        return funcao(*args, **kwargs)
    return nova_funcao

@requer_autenticacao
def painel_admin():
    print("Bem-vindo ao painel admin!")

----------------------------------------------------------------------------------------------------

## ***5. Decoradores de Classe***

### ***5.1. Decorador aplicando a classes***

In [ ]:
def log_inicializacao(cls):
    class ClsDecorada(cls):
        def __init__(self, *args, **kwargs):
            print(f"Inicializando {cls.__name__}")
            super().__init__(*args, **kwargs)
    return ClsDecorada

@log_inicializacao
class Pessoa:
    def __init__(self, nome):
        self.nome = nome

p = Pessoa("Ana") 

Inicializando Pessoa


### ***5.2. Método decorador (como @staticmethod, @classmethod)***

In [12]:
def meu_metodo_decorador(metodo):
    def nova_metodo(*args, **kwargs):
        print(f"Chamando {metodo.__name__}")
        return metodo(*args, **kwargs)
    return nova_metodo

class Calculadora:
    @meu_metodo_decorador
    def soma(self, a, b):
        return a + b

calc = Calculadora()
calc.soma(2, 3)

Chamando soma


5

----------------------------------------------------------------------------------------------------

## ***6. Decoradores usando classes (objetos chamáveis)***

In [13]:
class DecoradorPorClasse:
    def __init__(self, funcao):
        self.funcao = funcao
        self.contador = 0
    
    def __call__(self, *args, **kwargs):  
        self.contador += 1
        print(f"Called {self.contador} vezes")
        return self.funcao(*args, **kwargs)

@DecoradorPorClasse
def saudacao():
    print("Olá!")

saudacao()  
saudacao()  

Called 1 vezes
Olá!
Called 2 vezes
Olá!


----------------------------------------------------------------------------------------------------

## ***7. Problemas comuns e soluções***

### ***7.1. Perda do nome da função original***

In [ ]:
# Um problema de exemplo:
def decorador(funcao):
    def nova_funcao():
        return funcao()
    return nova_funcao

@decorador
def minha_fun():
    pass

print(minha_fun.__name__)  # "nova_funcao" -> Erro

nova_funcao


In [ ]:
# Solução: usar functools.wraps
from functools import wraps

def decorador_correto(funcao):
    @wraps(funcao)  # mantém nome, docstring, etc.
    def nova_funcao(*args, **kwargs):
        return funcao(*args, **kwargs)
    return nova_funcao

@decorador_correto
def minha_fun():
    pass

print(minha_fun.__name__)  # "minha_fun" -> correto

minha_fun


### ***7.2. Decoradores empilhados (ordem importante!)***

In [17]:
def decorador_a(funcao):
    @wraps(funcao)
    def nova():
        print("A - Antes")
        funcao()
        print("A - Depois")
    return nova

def decorador_b(funcao):
    @wraps(funcao)
    def nova():
        print("B - Antes")
        funcao()
        print("B - Depois")
    return nova

@decorador_a
@decorador_b
def minha_fun():
    print("Função original")

minha_fun()

A - Antes
B - Antes
Função original
B - Depois
A - Depois


Ordem: os decoradores são aplicados de baixo para cima

----------------------------------------------------------------------------------------------------

## ***8. Decoradores nativos do Python***

| Decorador              | Uso                                                |
| ---------------------- | -------------------------------------------------- |
| @staticmethod          | Método estático da classe                          |
| @classmethod           | Método que recebe a classe como primeiro argumento |
| @property              | Cria atributo tipo property (getter)               |
| @functools.lru_cache() | Cache automático com limite                        |
| @dataclasses.dataclass | Gera métodos automaticamente para classes          |

In [18]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

----------------------------------------------------------------------------------------------------

## ***9. Quando usar decoradores?***

- Logging de chamadas de função

- Cache/memoização de resultados

- Validação de argumentos

- Autenticação e controle de acesso

- Temporização de performance

- Retry automático de falhas

- Registro de funções (como no Flask)

Decoradores são uma das features mais elegantes do Python para metaprogramação. Eles permitem adicionar funcionalidades de maneira limpa e reutilizável